# Welcome to AI Summer School!

## Read this first — how to use this notebook

This is a **Jupyter notebook**: a document that mixes explanation and live code. You don't need to write code today. Here's all you need to know:

| Block type | What it looks like | What to do |
|---|---|---|
| **Explanation** (like this one) | White background, plain text | Just read it |
| **Code block** | Grey background, monospace font | Click the **▶** play button — the computer does the work |
| **"Your turn" section** | Labelled clearly in the text | Change one word or phrase, then press play |

**Run cells from top to bottom.** Each one builds on the last.

---

## What you'll see today

You're about to run real AI models — the same families of technology behind tools you use every day. Five demos:

1. **Sentiment Analysis** — how AI reads emotion in text (powers content moderation, customer feedback tools)
2. **Object Detection** — how AI finds objects in images (self-driving cars, Google Photos)
3. **Neural Style Transfer** — the 2016 technique that eventually led to Midjourney and DALL-E
4. **Text Similarity** — how AI compares meaning, not just words (the core of modern search)
5. **LLMs in Action** — zero-shot classification: AI reasoning about categories it was never trained on

You don't need to understand the code. The focus is on what the AI can and can't do — and why.

## Running in Google Colab

Google Colab is a free cloud environment — Python running in your browser, no installation needed on your machine.

### Three things to know:
1. **Run cells in order** — click ▶ or press `Shift + Enter`
2. **Wait for the spinning circle to stop** before moving to the next cell
3. **Yellow warning messages are fine. Red errors are problems.** If you see a red error, check you ran all the cells in order from the top.

### If something goes wrong:
Go to **Runtime → Restart session and run all** — this resets everything and starts fresh.

---

## Setup — Run Once and Move On

The next two cells install and load the AI libraries we need. Think of it as the game loading screen — necessary but not interesting.

**What to do**: press ▶ on each of the next two cells and wait. You'll see text scroll by. That's normal. Look for **"Setup complete."** before continuing.

> On Google Colab, this takes about 2 minutes the first time. After that it's cached.

In [ ]:
# Install required packages — run this cell once and ignore the output
print("Installing AI libraries... (~2 minutes on first run)")

!pip install -q transformers torch tensorflow-hub opencv-python-headless
!pip install -q --upgrade tensorflow

print("\nDone. Libraries installed.")

### Loading the libraries:

The cell below imports all the tools we'll use today. Run it and look for **"Setup complete."** at the bottom.

In [2]:
# Environment check - verify we're running in Google Colab
try:
    import google.colab
    print("Running in Google Colab environment")
    
    import tensorflow as tf
    # Check for GPU availability
    if tf.config.list_physical_devices('GPU'):
        print("GPU acceleration available")
    else:
        print("Using CPU (still suitable for this demonstration)")
    
except ImportError:
    print("Running in standard Jupyter environment")
    print("Consider using Google Colab for optimal performance")

print("\nEnvironment setup complete")

Running in standard Jupyter environment
Consider using Google Colab for optimal performance

Environment setup complete


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from PIL import Image
from io import BytesIO
import cv2

import tensorflow as tf
import tensorflow_hub as hub
import torch
from transformers import pipeline, BertModel, BertTokenizer
from sklearn.metrics.pairwise import cosine_similarity

# Handle Google Colab vs standard Jupyter environments
try:
    from google.colab.patches import cv2_imshow
    IN_COLAB = True
    print("Google Colab environment detected")
except ImportError:
    def cv2_imshow(img):
        plt.figure(figsize=(10, 8))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()
    IN_COLAB = False
    print("Standard Jupyter environment detected")

if tf.config.list_physical_devices('GPU'):
    print("GPU available — faster processing")
else:
    print("Using CPU — fine for today's demonstrations")

print("\nSetup complete.")

## Section 1: Sentiment Analysis
**Understanding Emotions in Text**

Sentiment analysis is a fundamental NLP task that determines the emotional tone of text. It's widely used in business applications like customer feedback analysis, social media monitoring, and market research.

We'll use a pre-trained model to analyse various text samples and observe its performance.

In [4]:
# Load pre-trained sentiment analysis model
print("Loading sentiment analysis model...")
sentiment_analysis = pipeline("sentiment-analysis")
print("Model loaded successfully")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading sentiment analysis model...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use mps:0


Model loaded successfully


In [5]:
# Sample texts with varying sentiment
texts = [
    "I absolutely love this sunny weather!",
    "This traffic jam is incredibly frustrating",
    "The movie was okay, nothing special",
    "I'm feeling a bit down today",
    "This pizza is amazing!",
    "I can't believe my team won the championship!"
]

print("Sample texts for analysis:")
for i, text in enumerate(texts, 1):
    print(f"{i}. {text}")

Sample texts for analysis:
1. I absolutely love this sunny weather!
2. This traffic jam is incredibly frustrating
3. The movie was okay, nothing special
4. I'm feeling a bit down today
5. This pizza is amazing!
6. I can't believe my team won the championship!


In [ ]:
# Analyse sentiment for each text sample
print("Sentiment Analysis Results:")
print("=" * 50)

results = sentiment_analysis(texts)

# Display results with confidence scores
for text, result in zip(texts, results):
    sentiment_icon = "+" if result['label'] == 'POSITIVE' else "-"
    confidence = result['score'] * 100
    
    print(f"{sentiment_icon} \"{text}\"")
    print(f"   Prediction: {result['label']} (Confidence: {confidence:.1f}%)")
    print("-" * 40)

Sentiment Analysis Results:
+ "I absolutely love this sunny weather!"
   Prediction: POSITIVE (Confidence: 100.0%)
----------------------------------------
- "This traffic jam is incredibly frustrating"
   Prediction: NEGATIVE (Confidence: 99.9%)
----------------------------------------
- "The movie was okay, nothing special"
   Prediction: NEGATIVE (Confidence: 99.5%)
----------------------------------------
- "I'm feeling a bit down today"
   Prediction: NEGATIVE (Confidence: 99.9%)
----------------------------------------
+ "This pizza is amazing!"
   Prediction: POSITIVE (Confidence: 100.0%)
----------------------------------------
+ "I can't believe my team won the championship!"
   Prediction: POSITIVE (Confidence: 99.9%)
----------------------------------------
+ "I absolutely love this sunny weather!"
   Prediction: POSITIVE (Confidence: 100.0%)
----------------------------------------
- "This traffic jam is incredibly frustrating"
   Prediction: NEGATIVE (Confidence: 99.9%)
--

### What just happened?

The model read each sentence and produced a probability score. It wasn't given a dictionary of "positive" and "negative" words — it was **trained on millions of labelled examples** (product reviews, social media posts, film reviews) until it learned which patterns of language tend to mean each sentiment.

The **confidence score** is almost always above 99% on clear sentences — the model has seen millions of nearly identical examples. It gets much less certain on ambiguous text. The challenge below will help you find those edges.

> The same technology runs at scale in: content moderation on social platforms, customer feedback analysis at companies, and financial market sentiment monitoring.

In [ ]:
# Interactive testing - try your own text
# Modify the text below to test different examples

your_text = "Replace this with your own message to test"

# Analyse your custom text
your_result = sentiment_analysis([your_text])[0]
sentiment_icon = "+" if your_result['label'] == 'POSITIVE' else "-"
confidence = your_result['score'] * 100

print("Your Custom Text Analysis:")
print(f"{sentiment_icon} \"{your_text}\"")
print(f"   Prediction: {your_result['label']} (Confidence: {confidence:.1f}%)")
print("\nTry modifying the text above to test different phrases and expressions.")

### Challenge 1: Testing Model Limitations

**Objective**: Explore edge cases where sentiment analysis might struggle.

**Try these examples**:
- Sarcastic statements: "Oh great, another Monday..."
- Mixed emotions: "I'm happy it's over but sad to leave"
- Context-dependent phrases: "This is sick!" (slang vs. literal)
- Cultural references or idioms

**Discussion points**:
- How do confidence scores relate to your intuitive assessment?
- What types of text seem to confuse the model?
- How might training data bias affect results? 

## Section 2: Object Detection — Computers That Can See

Your phone recognises your face to unlock. Google Photos knows which of your pictures contain your dog. Self-driving cars identify pedestrians in real time. All of these rely on **object detection** — a neural network that scans an image and draws boxes around things it recognises.

We'll run a model trained on the COCO dataset (80 everyday object categories). You'll see it annotate an image with what it found and how confident it is about each detection.

In [ ]:
# Utility function for downloading and displaying images
def display_image(url, title="Image"):
    try:
        print(f"Downloading image from: {url}")
        headers = {"User-Agent": "Mozilla/5.0 (Colab fetch)"}      # ← added
        response = requests.get(url, headers=headers, timeout=15)  # ← UA + timeout
        response.raise_for_status()                                # ← surfaces 4xx/5xx early
        img = Image.open(BytesIO(response.content)).convert("RGB")
        
        # Display the image
        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.title(title, fontsize=14)
        plt.axis('off')
        plt.show()
        
        print(f"Image loaded - Size: {img.size}")
        return img, np.array(img)
    except Exception as e:
        print(f"Error loading image: {e}")
        return None, None

In [ ]:
# Load pre-trained object detection model
print("Loading object detection model...")
print("This model was trained on the COCO dataset with 80+ object categories")
detector = hub.load("https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2")
print("Model loaded successfully")

In [ ]:
# Select test image - urban street scene
image_url = "https://images.unsplash.com/photo-1449824913935-59a10b8d2000?ixlib=rb-4.0.3&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D&auto=format&fit=crop&w=1000&q=80"

print("Test image: Urban street scene")

In [ ]:
# Load and display the test image
img, imgarray = display_image(image_url, "Original Image")

In [ ]:
# Prepare image for model input
print("Preprocessing image for analysis...")
img = img.resize((640, 480))  # Resize to model's expected input size
img_tensor = tf.convert_to_tensor(np.array(img), dtype=tf.uint8)
img_tensor = tf.expand_dims(img_tensor, axis=0)  # Add batch dimension
print("Image preprocessing complete")

In [ ]:
# Run object detection inference
print("Running object detection...")
print("Identifying objects and their locations...")
result = detector(img_tensor)
print("Detection complete")

In [ ]:
# COCO dataset class labels — the 80 categories this model was trained to recognise
COCO_LABELS = {
    1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorcycle', 5: 'airplane',
    6: 'bus', 7: 'train', 8: 'truck', 9: 'boat', 10: 'traffic light',
    11: 'fire hydrant', 13: 'stop sign', 14: 'parking meter', 15: 'bench',
    16: 'bird', 17: 'cat', 18: 'dog', 19: 'horse', 20: 'sheep', 21: 'cow',
    22: 'elephant', 23: 'bear', 24: 'zebra', 25: 'giraffe', 27: 'backpack',
    28: 'umbrella', 31: 'handbag', 32: 'tie', 33: 'suitcase', 34: 'frisbee',
    35: 'skis', 36: 'snowboard', 37: 'sports ball', 38: 'kite',
    39: 'baseball bat', 40: 'baseball glove', 41: 'skateboard', 42: 'surfboard',
    43: 'tennis racket', 44: 'bottle', 46: 'wine glass', 47: 'cup',
    48: 'fork', 49: 'knife', 50: 'spoon', 51: 'bowl', 52: 'banana',
    53: 'apple', 54: 'sandwich', 55: 'orange', 56: 'broccoli', 57: 'carrot',
    58: 'hot dog', 59: 'pizza', 60: 'donut', 61: 'cake', 62: 'chair',
    63: 'couch', 64: 'potted plant', 65: 'bed', 67: 'dining table',
    70: 'toilet', 72: 'tv', 73: 'laptop', 74: 'mouse', 75: 'remote',
    76: 'keyboard', 77: 'cell phone', 78: 'microwave', 79: 'oven',
    80: 'toaster', 81: 'sink', 82: 'refrigerator', 84: 'book', 85: 'clock',
    86: 'vase', 87: 'scissors', 88: 'teddy bear', 89: 'hair drier',
    90: 'toothbrush'
}

def draw_boxes(image, boxes, class_ids, scores, max_boxes=15, min_score=0.4):
    colors = list(plt.cm.tab20(np.linspace(0, 1, 20)))
    font = cv2.FONT_HERSHEY_SIMPLEX
    detected = []

    for i in range(min(max_boxes, boxes.shape[0])):
        if scores[i] >= min_score:
            class_id = int(class_ids[i])
            label = COCO_LABELS.get(class_id, f'unknown ({class_id})')
            confidence = scores[i] * 100

            color_rgb = colors[class_id % len(colors)][:3]
            color_bgr = tuple(int(c * 255) for c in reversed(color_rgb))

            ymin, xmin, ymax, xmax = tuple(boxes[i])
            h, w = image.shape[:2]
            x1, y1 = int(xmin * w), int(ymin * h)
            x2, y2 = int(xmax * w), int(ymax * h)

            cv2.rectangle(image, (x1, y1), (x2, y2), color_bgr, 2)
            cv2.putText(image, f"{label} {confidence:.0f}%",
                        (x1, max(y1 - 8, 15)), font, 0.5, color_bgr, 2)
            detected.append(f"{label} ({confidence:.0f}% confident)")

    return image, detected

In [ ]:
# Extract detection results
boxes = result["detection_boxes"][0].numpy()
class_names = result["detection_classes"][0].numpy()
scores = result["detection_scores"][0].numpy()

In [ ]:
# Draw labelled bounding boxes on the image
output_img, detected_objects = draw_boxes(np.array(img), boxes, class_names, scores)

In [ ]:
print("Objects detected:")
for obj in detected_objects:
    print(f"  {obj}")
print()

cv2_imshow(output_img)

### What just happened?

The model scanned the image by dividing it into a grid and asking, for each region: *is there an object here, and if so, what is it?* It was trained on the **COCO dataset** — 330,000 images with 1.5 million manually drawn bounding boxes across 80 categories, labelled by human annotators.

Notice it can only detect things it was trained on. Show it an object outside those 80 categories and it either ignores it or guesses something nearby. That's a core limitation of supervised learning: the model is bounded by what was in its training data.

> The same family of models runs in: self-driving car perception systems, phone cameras (auto-focus, portrait mode), Google Photos auto-tagging, and retail shelf stock monitoring.

### Challenge 2: Model Robustness Testing

**Objective**: Test the object detection model with challenging scenarios.

**Suggested test cases**:
- Crowded scenes with overlapping objects
- Unusual viewing angles or lighting conditions  
- Abstract or artistic images
- Very small or very large scale objects
- Partially occluded objects

**Image search suggestions**:
- "crowded marketplace"
- "abstract geometric art"
- "extreme close-up photography"
- "low light photography"

**Analysis questions**:
- What object categories does the model handle well/poorly?
- How does object size affect detection accuracy?
- What are the implications for real-world applications like autonomous vehicles? 

## Section 3: Neural Style Transfer — The Ancestor of AI Art

In 2015, researchers published *"A Neural Algorithm of Artistic Style"* — showing you could mathematically separate the *content* of an image (what's in it) from its *style* (how it's painted) and recombine them at will.

Take a photo of a street; ask a neural network to repaint it in Van Gogh's brushstroke style. The paper was published in September 2015, and by 2016 an app called Prisma made it go viral on Instagram.

**Why this matters now**: the insight that neural networks encode "style" as a separable mathematical property directly influenced the development of Stable Diffusion, Midjourney, and DALL-E. We're running the original 2015 technique below. The results look noticeably different from a modern image generator — that gap represents about eight years of research.

**How it works in one sentence**: two competing objectives (preserve the photo's content structure; match the painting's texture patterns) are optimised simultaneously by adjusting every pixel until a compromise image satisfies both.

In [ ]:
# Load AI artist model
print("🎨 Loading AI art studio...")
print("   This AI learned from thousands of famous paintings!")
style_transfer_model = hub.load("https://tfhub.dev/google/magenta/arbitrary-image-stylization-v1-256/2")
print("✅ AI artist ready to create masterpieces!")

In [ ]:
# Choose our content image (what we want to paint) and style image (how we want to paint it)
content_image_url = "https://images.unsplash.com/photo-1535930891776-0c2dfb7fda1a?ixlib=rb-4.0.3&auto=format&fit=crop&w=400&q=80"
style_image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/400px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"

print("🖼️  Content Image: A beautiful landscape")
print("🎨 Style Image: Van Gogh's 'Starry Night'")

In [ ]:
# Let's see our source images
print("📷 CONTENT IMAGE (what to paint):")
content_img, content_array = display_image(content_image_url, "Content Image")

print("\n🎨 STYLE IMAGE (how to paint it):")
style_img, style_array = display_image(style_image_url, "Style Image - Van Gogh's Starry Night")

In [ ]:
print("Preparing images for style transfer...")

if content_array is None or style_array is None:
    print("\nAn image failed to download — this is usually a temporary network issue.")
    print("Fix: re-run the cell above to retry, or replace the URL with any image link.")
    raise ValueError("Image not available. Re-run the cell above first.")

content_image = tf.convert_to_tensor(content_array, dtype=tf.float32)
content_image = tf.image.resize(content_image, (256, 256)) / 255.0
content_image = tf.expand_dims(content_image, axis=0)

style_image = tf.convert_to_tensor(style_array, dtype=tf.float32)
style_image = tf.image.resize(style_image, (256, 256)) / 255.0
style_image = tf.expand_dims(style_image, axis=0)

print("Images ready.")

In [ ]:
# 🎨 The magic happens here! AI creates art!
print("🤖 AI is painting... combining content with Van Gogh's style...")
stylized_image = style_transfer_model(tf.constant(content_image), tf.constant(style_image))[0]
print("✅ Masterpiece created!")

In [ ]:
# 🖼️ Reveal the AI artwork!
plt.figure(figsize=(10, 8))
plt.imshow(stylized_image[0])
plt.title('🎨 AI-Generated Artwork\n(Original photo painted in Van Gogh style!)', fontsize=16)
plt.axis('off')
plt.show()

print("🎉 Amazing! The AI combined your photo with Van Gogh's painting style!")
print("💡 Try different images and see what the AI creates!")

### What just happened?

The model ran an **optimisation loop** — it started with the content image and repeatedly adjusted every pixel, trying to simultaneously satisfy two objectives:
1. Keep the overall shapes and layout of the original photo (content loss)
2. Match the texture and colour statistics of the Van Gogh painting (style loss)

It has no understanding of art. It matched patterns in the neural network's internal representations of the two images.

**Compare this to Midjourney or DALL-E**: those models were trained on billions of image-text pairs and generate images from random noise, guided by a text description. The quality leap is dramatic — but the insight that neural networks encode "style" as a separable mathematical property traces directly back to this 2015 paper. You just ran the original.

### Challenge 3: From 2015 to Now

You just ran the original technique. Now go compare it to what exists today.

**Step 1**: Go to any modern image generator — Midjourney, DALL-E (in ChatGPT), Adobe Firefly, or Stable Diffusion. Generate an image using a prompt that combines content and style, for example:
> *"A street scene in London, painted in the style of Van Gogh's Starry Night"*

**Step 2**: Put the two results side by side (the style transfer output above, your generated image). What's different?

**Things to notice**:
- Sharpness and coherence of the output
- How faithfully it captured the artistic style
- How much control you had over the result
- How long each took

**Discussion**:
- The style transfer above took researchers months to develop in 2015. The modern equivalent took you 10 seconds to type. What changes when a capability becomes instant and free?
- Modern generators were trained on billions of images scraped from the internet — including work by living artists. The style transfer model wasn't. Does that distinction matter?
- If AI can replicate any artistic style on demand, what is the value of developing a distinctive visual style as a human artist?

## Section 4: Semantic Text Similarity — Meaning, Not Just Words

A basic word-matching search would say "I love dogs" and "I adore canines" have nothing in common — they share no words. But they mean the same thing. **Semantic similarity** is how AI measures meaning directly, matching concepts rather than characters.

This is what makes modern search actually useful: you can describe something vaguely and still find it.

We'll use BERT — the model that transformed NLP in 2018 and still underpins much of modern search — to compute similarity scores between sentence pairs. A score near 1.0 means nearly identical meaning; near 0.0 means unrelated.

In [ ]:
# Load BERT model for semantic similarity
print("Loading BERT language model...")
print("BERT (Bidirectional Encoder Representations from Transformers)")
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
print("BERT model loaded successfully")

In [ ]:
def get_sentence_embedding(sentence):
    # Tokenize the input sentence
    inputs = tokenizer(sentence, return_tensors='pt', truncation=True, padding=True, max_length=512)
    # Get the hidden states from the model
    with torch.no_grad():
        outputs = model(**inputs)
    # Take the mean of the token embeddings from the last hidden state
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.numpy()

In [ ]:
def compute_similarity(sentence1, sentence2):
    # Get sentence embeddings for both sentences
    embedding1 = get_sentence_embedding(sentence1)
    embedding2 = get_sentence_embedding(sentence2)
    # Compute cosine similarity between the embeddings
    similarity = cosine_similarity(embedding1, embedding2)[0][0]
    return similarity

In [ ]:
sentence_pairs = [
    ("AI is transforming the world.", "Artificial intelligence is changing everything."),
    ("The cat is sleeping peacefully.", "A feline is napping quietly."),
    ("I love pizza!", "Pizza is terrible."),
    ("The weather is sunny today.", "Today is a bright, clear day."),
    ("Programming is fun!", "I hate vegetables.")
]

print("Sentence pairs to compare:")
for i, (s1, s2) in enumerate(sentence_pairs, 1):
    print(f"\n{i}. A: '{s1}'")
    print(f"   B: '{s2}'")

In [ ]:
# Let's see how similar each pair is!
print("\n🤖 AI Similarity Analysis Results:")
print("=" * 50)

for i, (sentence1, sentence2) in enumerate(sentence_pairs, 1):
    similarity_score = compute_similarity(sentence1, sentence2)
    
    # Add interpretation
    if similarity_score > 0.8:
        interpretation = "Very Similar! 🎯"
    elif similarity_score > 0.6:
        interpretation = "Quite Similar 👍"
    elif similarity_score > 0.4:
        interpretation = "Somewhat Similar 🤔"
    else:
        interpretation = "Not Very Similar 🚫"
    
    print(f"\n{i}. Pair {i}:")
    print(f"   A: '{sentence1}'")
    print(f"   B: '{sentence2}'")
    print(f"   Similarity: {similarity_score:.3f} - {interpretation}")

print("\n💡 1.0 means identical, 0.0 means completely different!")

### What just happened?

BERT converted each sentence into a list of 768 numbers — a **vector** in high-dimensional space that encodes its meaning. The similarity score is the cosine of the angle between the two vectors. Sentences that mean similar things end up pointing in similar directions, even if they share no words.

BERT learned to do this by training on billions of words, predicting masked words and whether two sentences followed each other. It was never told "these sentences are similar" — it learned *meaning* as a side-effect of learning to predict language.

> The same approach powers: Google Search (matching queries to documents by meaning, not keywords), Spotify and Netflix recommendations, email spam filters, and the "find related papers" feature in academic databases.

In [ ]:
# --- YOUR TURN ---
# Change these two sentences and run the cell

your_sentence1 = "Change this to your first sentence"
your_sentence2 = "Change this to your second sentence"

your_similarity = compute_similarity(your_sentence1, your_sentence2)

if your_similarity > 0.8:
    label = "Very Similar"
elif your_similarity > 0.6:
    label = "Quite Similar"
elif your_similarity > 0.4:
    label = "Somewhat Similar"
else:
    label = "Not Very Similar"

print(f"A: '{your_sentence1}'")
print(f"B: '{your_sentence2}'")
print(f"Score: {your_similarity:.3f} — {label}")
print("\nChallenge: can you find two sentences that mean the same thing but share no words?")

### Challenge 4: The Similarity Score Olympics!

**Challenges to try**:
1. **The synonym test**: do synonyms always get high scores? Try "big" vs "large", "happy" vs "joyful"
2. **The translation test**: compare "Hello" with "Hola" or another language you know
3. **The context test**: do sentences with the same words but different meanings confuse the model?

**Examples to explore**:
- "The bank was closed" vs "The river bank was muddy"
- "Time flies like an arrow" vs "Fruit flies like a banana"
- "I saw her duck" vs "I saw her duck under the table"

**Discussion**:
- When does the AI get similarity "wrong" in your opinion?
- How might cultural context affect similarity scores?
- What are the limits of measuring meaning as a number?

## Section 5: LLMs in Action — Zero-Shot Classification

**The technology behind ChatGPT, Claude, and Gemini**

The previous demos used models each trained for one specific task: sentiment analysis, object detection, style transfer. Each model does one job.

Modern large language models (LLMs) are different. They're trained on so much text — essentially a large portion of the internet — that they develop a general understanding of concepts and relationships. They can then apply that understanding to tasks they were never explicitly taught.

**Zero-shot classification** demonstrates this clearly: you give the model a piece of text and a list of categories *you just invented*, and it figures out which one fits best. No examples, no training on those categories — just the model's general understanding of language.

This is the same family of capability that makes ChatGPT useful across writing, coding, analysis, and conversation without being fine-tuned for each task separately.

In [ ]:
print("Loading language model... (may take a minute on first run)")

zero_shot_classifier = pipeline("zero-shot-classification")

print("Model ready.\n")

# The model classifies this sentence into categories we just made up —
# categories it was never explicitly trained on

text = "I stayed up until 2am finishing the series and now I can't focus in lectures"

categories = [
    "sleep deprivation",
    "academic struggle",
    "entertainment addiction",
    "relatable student experience"
]

result = zero_shot_classifier(text, candidate_labels=categories)

print(f'Text: "{text}"\n')
print("How the model classifies this (into categories it was never trained on):")
for label, score in zip(result['labels'], result['scores']):
    bar = "█" * int(score * 40)
    print(f"  {label:<35} {bar} {score*100:.1f}%")

In [ ]:
# Same text — two completely different sets of categories
# Watch how the model's interpretation shifts depending on what you ask it to look for

text = "The new iPhone has an incredible camera but the battery drains by 3pm every day"

categories_v1 = ["product praise", "product criticism", "balanced review", "off-topic"]
result1 = zero_shot_classifier(text, candidate_labels=categories_v1)

categories_v2 = ["frustration", "excitement", "indifference", "regret"]
result2 = zero_shot_classifier(text, candidate_labels=categories_v2)

print(f'Text: "{text}"\n')
print("With product-focused categories:")
for label, score in zip(result1['labels'], result1['scores']):
    bar = "█" * int(score * 35)
    print(f"  {label:<22} {bar} {score*100:.1f}%")

print("\nWith emotion-focused categories:")
for label, score in zip(result2['labels'], result2['scores']):
    bar = "█" * int(score * 35)
    print(f"  {label:<22} {bar} {score*100:.1f}%")

print("\nThe text hasn't changed. The question has. This is prompt sensitivity.")

In [ ]:
# --- YOUR TURN ---
# Change the text and categories below, then run the cell

my_text = "I've been using the gym every morning this week and I feel so much better"

my_categories = [
    "health and fitness",
    "mental wellbeing",
    "building habits",
    "social media caption"
]

result = zero_shot_classifier(my_text, candidate_labels=my_categories)

print(f'Your text: "{my_text}"\n')
print("How the model reads it:")
for label, score in zip(result['labels'], result['scores']):
    bar = "█" * int(score * 40)
    print(f"  {label:<28} {bar} {score*100:.1f}%")

print("\nTry changing the categories to completely different ones.")
print("Watch how the scores shift depending on what you ask the model to look for.")

### What just happened?

The model classified text into categories it was **never explicitly trained on**. You didn't give it examples — you just described the categories in plain English, and it figured out which one fits best.

This is called **zero-shot classification**, and it works because large language models, trained on enormous amounts of text, develop a rich internal representation of concepts and their relationships — rich enough to reason about new categories on the fly.

**The prompt sensitivity experiment** reveals something important: the model doesn't have a fixed "opinion" about a piece of text. Its output depends entirely on what you ask it to look for. This is why careful prompt design matters so much when using AI tools — you're not retrieving a stored fact, you're shaping how the model frames the problem.

> This same capability is what makes modern LLMs flexible: a model trained once can answer questions about topics that didn't exist when it was trained, because it learned to reason about language itself, not just a fixed list of tasks.

## The AI Landscape in 2026

The field has moved fast. Here's a snapshot of the major categories and tools as of mid-2026 — many of which you probably already use.

### Text & Conversation
| Tool | What it does |
|---|---|
| **Claude** (Anthropic) | Long-context reasoning, coding, analysis |
| **ChatGPT / GPT-4o** (OpenAI) | Multimodal chat, image understanding, voice |
| **Gemini 2.0** (Google) | Integrated with Google Workspace and Search |
| **Llama 3** (Meta) | Open-source model — runs locally on your machine |

### Image Generation
| Tool | What it does |
|---|---|
| **Midjourney V7** | High-quality artistic generation from text prompts |
| **DALL-E 3 / GPT-4o** | Text-to-image, built into ChatGPT |
| **Stable Diffusion 3** | Open-source, fully customisable, runs locally |
| **Adobe Firefly** | Commercially safe generation, in Creative Cloud |

### Video & Audio
| Tool | What it does |
|---|---|
| **Sora** (OpenAI) | Text-to-video generation |
| **Runway Gen-3** | AI video editing and scene generation |
| **Suno / Udio** | Full-track music generation from text prompts |
| **ElevenLabs** | Voice synthesis and cloning |

### Coding & Productivity
| Tool | What it does |
|---|---|
| **GitHub Copilot** | Code completion and generation in your IDE |
| **Cursor / Windsurf** | AI-native code editors |
| **Claude Code** | Agentic coding from the terminal |
| **NotebookLM** (Google) | AI analysis and Q&A over your own documents |

---

### The pace of change

The tools on this list will look dated by 2027. The underlying *techniques* — transformers, diffusion models, RLHF — are more stable. Understanding the fundamentals this week means you can evaluate whatever appears next.

### Exploration: Prompt Engineering in 10 Minutes

The most immediately useful skill when working with any of the tools above is **prompt engineering** — how you phrase a request dramatically changes what you get back.

**Try this** (pick any chatbot — Claude, ChatGPT, Gemini):

1. **Vague prompt**: *"Explain neural networks"*
2. **Better prompt**: *"Explain neural networks in two sentences, for a first-year university student with no maths background"*
3. **Structured prompt**: *"Explain neural networks using a single analogy. Then give one real-world example. Keep it under 100 words."*

Run all three on the same tool and compare the outputs.

**Then try**:
- Ask it to explain one of today's demos in plain English
- Ask it to critique a piece of your own writing
- Ask it to generate three variations of an idea you have

**Discussion**: Where did precise prompting help most? Where did the output barely change? What does that tell you about what the model is actually doing?

---

### Bigger questions to sit with

- Which of today's five demos surprised you most — and why?
- What happens to professions (medicine, law, creative work, software) when tools like these are free and instant?
- We saw models fail or behave unexpectedly. What would it take for you to *trust* an AI system in a high-stakes decision?
- Who benefits from AI moving this fast? Who bears the cost?

## What comes next

### Course overview

Today's session introduced the range of what AI can do. The rest of the course builds up the tools to understand *how*:

- **Day 2**: Python and data fundamentals
- **Day 3**: Supervised learning — training models from labelled data
- **Day 4**: Unsupervised learning — finding patterns without labels
- **Day 5**: Neural networks — the architecture behind everything you saw today
- **Day 6**: Natural language processing — building your own text models
- **Day 8**: Reinforcement learning — AI that learns from feedback
- **Day 10**: Modern frameworks and putting it all together

### Saving your work

In **Google Colab**: File → Save a copy in Drive. Your changes are saved to your Google account.

---

### One thing to take away

Every model you ran today was trained on data collected and labelled by humans. Its capabilities are defined by that data — what was included, what was left out, and whose perspective shaped the labels. As you learn to build these systems, that context matters as much as the code.